# Stable Diffusion 模型结构全览（基于 HuggingFace Diffusers）

本 Notebook 展示 Stable Diffusion 各核心子模块的网络结构，涵盖：
- **VAE（变分自编码器）**：负责图像的压缩编码与像素空间解码
- **CLIP 文本编码器**：将文本 prompt 编码为语义向量
- **CLIPTokenizer**：将文本转化为 token id 序列
- **U-Net（去噪网络）**：在 latent 空间中进行条件去噪
- **时间步嵌入函数**：将扩散时间步编码为向量，注入 U-Net

所有结构均来自 `stabilityai/stable-diffusion-2-1-base`，通过 `print(model)` 打印获得。

## 一、VAE（变分自编码器）结构

`AutoencoderKL` 是 Stable Diffusion 的图像编解码器，包含两个子模块：

| 子模块 | 功能 | 输入 → 输出 |
|--------|------|-------------|
| `Encoder` | 将原始图像压缩到 latent 空间 | `[B,3,512,512]` → `[B,4,64,64]` |
| `Decoder` | 将 latent 还原为像素图像 | `[B,4,64,64]` → `[B,3,512,512]` |

**编码器结构要点：**
- `conv_in`：将 3 通道 RGB 升至 128 通道
- `down_blocks`：4 个 DownEncoderBlock2D，通道从 128 → 256 → 512 → 512，每次下采样使空间尺寸减半（512→64）
- `mid_block`：UNetMidBlock2D，含 1 个自注意力层夹在 2 个 ResnetBlock 之间
- `conv_out`：输出 8 通道（VAE 会分为均值和方差各 4 通道，采样后取 4 通道 latent）

**解码器结构要点：**
- `conv_in`：将 4 通道 latent 升至 512 通道
- `up_blocks`：4 个 UpDecoderBlock2D，通道从 512 → 512 → 256 → 128，每次上采样使空间尺寸翻倍（64→512）
- `conv_out`：输出 3 通道 RGB 图像
- `quant_conv` / `post_quant_conv`：对 latent 做 1×1 卷积对齐，连接 Encoder 输出和 Decoder 输入

```python
AutoencoderKL(                                                                         # VAE 根类：变分自编码器，含 Encoder + Decoder 两个子模块
  (encoder): Encoder(                                                                  # ---- 编码器：将原始图像 [B,3,H,W] 压缩到 latent 空间 [B,4,H/8,W/8] ----
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))      # 输入卷积：3 通道 RGB 升至 128 通道；padding=1 保持空间尺寸不变（512×512 不变）
    (down_blocks): ModuleList(                                                          # 4 个下采样块，通道逐步升维：128→256→512→512，空间逐步降维：512→256→128→64
      (0): DownEncoderBlock2D(                                                          # 第 1 个下采样块：128 通道，含 2 个 ResnetBlock + 1 个 Downsample
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(                                                    # 两个残差块，通道数保持 128→128
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)                        # 第一个 GroupNorm：32 组，每组 128/32=4 个通道，稳定训练
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 3×3 卷积，same padding，空间尺寸不变
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)                        # 第二个 GroupNorm
            (dropout): Dropout(p=0.0, inplace=False)                                  # Dropout，训练时 p=0.0 表示未启用（推理无影响）
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 第二个 3×3 卷积
            (nonlinearity): SiLU()                                                     # Swish/SiLU 激活：SiLU(x)=x·σ(x)，比 ReLU 更平滑
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))                # stride=2 的卷积实现下采样：空间尺寸减半 512→256（无 padding 表示代码内部已处理）
          )
        )
      )
      (1): DownEncoderBlock2D(                                                          # 第 2 个下采样块：通道 128→256，空间 256→128
        (resnets): ModuleList(
          (0): ResnetBlock2D(                                                            # 第一个残差块：通道升维 128→256，有 conv_shortcut 做跳跃连接
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 升维卷积：128→256
            (norm2): GroupNorm(32, 256, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
            (conv_shortcut): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1))       # 1×1 跳跃连接：维度对齐（128→256），使残差路径通道匹配
          )
          (1): ResnetBlock2D(                                                            # 第二个残差块：通道保持 256→256，无 conv_shortcut
            (norm1): GroupNorm(32, 256, eps=1e-06, affine=True)
            (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 256, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2))                # 下采样：256→128
          )
        )
      )
      (2): DownEncoderBlock2D(                                                          # 第 3 个下采样块：通道 256→512，空间 128→64
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 256, eps=1e-06, affine=True)
            (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 升维：256→512
            (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
            (conv_shortcut): Conv2d(256, 512, kernel_size=(1, 1), stride=(1, 1))       # 跳跃连接：256→512
          )
          (1): ResnetBlock2D(
            (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
            (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(512, 512, kernel_size=(3, 3), stride=(2, 2))                # 下采样：64→32（注：VAE 编码器共 3 次下采样，512/8=64 latent）
          )
        )
      )
      (3): DownEncoderBlock2D(                                                          # 第 4 个下采样块：通道 512 保持，无 Downsample（已是最小分辨率 64）
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(                                                    # 两个 512→512 残差块，在最小分辨率处进一步提取特征
            (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
            (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
      )
    )
    (mid_block): UNetMidBlock2D(                                                        # 瓶颈层：2 个 ResnetBlock + 中间夹 1 个自注意力（在最小空间分辨率 64 处）
      (attentions): ModuleList(
        (0): Attention(                                                                  # 自注意力：在编码器最小分辨率（64×64）处建立全局像素间依赖
          (group_norm): GroupNorm(32, 512, eps=1e-06, affine=True)                     # 注意力前的 GroupNorm 归一化
          (to_q): Linear(in_features=512, out_features=512, bias=True)                 # 查询投影 Q：512→512
          (to_k): Linear(in_features=512, out_features=512, bias=True)                 # 键投影 K：512→512
          (to_v): Linear(in_features=512, out_features=512, bias=True)                 # 值投影 V：512→512
          (to_out): ModuleList(
            (0): Linear(in_features=512, out_features=512, bias=True)                  # 输出投影：512→512
            (1): Dropout(p=0.0, inplace=False)
          )
        )
      )
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(                                                      # 夹住注意力的两个残差块（结构：ResNet → Attention → ResNet）
          (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
          (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
    )
    (conv_norm_out): GroupNorm(32, 512, eps=1e-06, affine=True)                        # 编码器输出前的归一化
    (conv_act): SiLU()                                                                  # 编码器输出前的激活函数
    (conv_out): Conv2d(512, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))     # 输出 8 通道：前 4 通道为均值 μ，后 4 通道为对数方差 log(σ²)，重参数化采样得到 4 通道 latent
  )
  (decoder): Decoder(                                                                   # ---- 解码器：将 latent [B,4,H/8,W/8] 还原为图像 [B,3,H,W] ----
    (conv_in): Conv2d(4, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))      # 输入卷积：4 通道 latent 升至 512 通道
    (up_blocks): ModuleList(                                                             # 4 个上采样块，通道逐步降维：512→512→256→128，空间逐步升维：64→128→256→512
      (0-1): 2 x UpDecoderBlock2D(                                                     # 前两个上采样块：通道 512 保持，含 3 个 ResNet + Upsample
        (resnets): ModuleList(
          (0-2): 3 x ResnetBlock2D(                                                    # 三个 512→512 残差块（解码器 ResNet 数量比编码器多一个）
            (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
            (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (upsamplers): ModuleList(
          (0): Upsample2D(                                                              # 双线性插值上采样：空间尺寸翻倍（64→128→256）
            (conv): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 上采样后的 3×3 卷积，消除插值伪影
          )
        )
      )
      (2): UpDecoderBlock2D(                                                            # 第 3 个上采样块：通道 512→256，空间 256→512
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
            (conv1): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 降维卷积：512→256
            (norm2): GroupNorm(32, 256, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
            (conv_shortcut): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))       # 1×1 跳跃连接：维度对齐 512→256
          )
          (1-2): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 256, eps=1e-06, affine=True)
            (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 256, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (upsamplers): ModuleList(
          (0): Upsample2D(                                                              # 上采样：256→512
            (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
      )
      (3): UpDecoderBlock2D(                                                            # 第 4 个上采样块：通道 256→128，无 Upsample（已是最终分辨率）
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 256, eps=1e-06, affine=True)
            (conv1): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 降维：256→128
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
            (conv_shortcut): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))       # 跳跃连接：256→128
          )
          (1-2): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
      )
    )
    (mid_block): UNetMidBlock2D(                                                        # 解码器瓶颈层：结构与编码器 mid_block 相同（ResNet→Attention→ResNet）
      (attentions): ModuleList(
        (0): Attention(                                                                  # 解码器中的自注意力，在 latent 分辨率建立全局依赖
          (group_norm): GroupNorm(32, 512, eps=1e-06, affine=True)
          (to_q): Linear(in_features=512, out_features=512, bias=True)
          (to_k): Linear(in_features=512, out_features=512, bias=True)
          (to_v): Linear(in_features=512, out_features=512, bias=True)
          (to_out): ModuleList(
            (0): Linear(in_features=512, out_features=512, bias=True)
            (1): Dropout(p=0.0, inplace=False)
          )
        )
      )
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(
          (norm1): GroupNorm(32, 512, eps=1e-06, affine=True)
          (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (norm2): GroupNorm(32, 512, eps=1e-06, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
    )
    (conv_norm_out): GroupNorm(32, 128, eps=1e-06, affine=True)                        # 解码器输出前的归一化
    (conv_act): SiLU()                                                                  # 解码器输出前的激活函数
    (conv_out): Conv2d(128, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))     # 输出卷积：将 128 通道特征映射为 3 通道 RGB 图像（值域经 tanh 归一化到 [-1,1]）
  )
  (quant_conv): Conv2d(8, 8, kernel_size=(1, 1), stride=(1, 1))                       # 量化卷积（1×1）：对编码器输出的 8 通道做线性变换，分离均值/方差用于 KL 散度计算
  (post_quant_conv): Conv2d(4, 4, kernel_size=(1, 1), stride=(1, 1))                  # 反量化卷积（1×1）：将重参数化采样后的 4 通道 latent 线性投影后送入解码器
)
```

## 二、CLIP 文本编码器（CLIPTextModel）结构

`CLIPTextModel` 是基于 Transformer 的文本编码器，负责将文本 token 序列映射为语义向量。

**结构要点：**
- `token_embedding`：词表大小 49408，嵌入维度 1024，将每个 token id 映射为 1024 维向量
- `position_embedding`：最大序列长度 77，为每个位置学习一个 1024 维位置编码
- `encoder`：由 **23 个** `CLIPEncoderLayer` 堆叠而成，每层包含：
  - `self_attn`：多头自注意力（Q/K/V 均为 1024→1024），实现 token 间的信息交互
  - `layer_norm1/2`：两个 LayerNorm 层稳定训练
  - `mlp`：两层 FFN（1024→4096→1024），使用 GELU 激活
- `final_layer_norm`：最终层归一化，输出 [B, 77, 1024] 的文本特征

该特征随后送入 U-Net 各层的交叉注意力（Cross-Attention）中作为 K 和 V。

```python
CLIPTextModel(                                                                         # CLIP 文本编码器根类，封装 CLIPTextTransformer
  (text_model): CLIPTextTransformer(                                                   # 核心 Transformer 编码器，处理 token 序列
    (embeddings): CLIPTextEmbeddings(                                                  # 嵌入层：将 token id 和位置索引转换为连续向量
      (token_embedding): Embedding(49408, 1024)                                        # token 嵌入表：词表大小 49408，嵌入维度 1024（每个 token id 对应一个 1024 维向量）
      (position_embedding): Embedding(77, 1024)                                        # 位置嵌入表：最大序列长度 77，每个位置有一个可学习的 1024 维位置编码
    )
    (encoder): CLIPEncoder(                                                             # Transformer Encoder 主体，由多个 CLIPEncoderLayer 堆叠而成
      (layers): ModuleList(
        (0-22): 23 x CLIPEncoderLayer(                                                 # 共 23 层 Encoder Layer，对应 CLIP ViT-L/14 的文本编码器深度
          (self_attn): CLIPAttention(                                                  # 多头自注意力（Multi-Head Self-Attention）：允许每个 token 关注序列中所有其他 token
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)           # 键投影 K：1024→1024（多头，16 头 × 64 维/头）
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)           # 值投影 V：1024→1024
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)           # 查询投影 Q：1024→1024
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)         # 输出投影：将多头拼接后的结果映射回 1024 维
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)        # Pre-Norm LN1：自注意力前的层归一化（SD 2.x 使用 Pre-LN 结构）
          (mlp): CLIPMLP(                                                               # 前馈网络（FFN）：两层线性 + GELU 激活
            (activation_fn): GELUActivation()                                          # GELU 激活函数：比 ReLU 更平滑，适合 Transformer 的 FFN
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)              # 第一层全连接：升维 1024→4096（4× 扩展因子），提取更丰富的特征
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)              # 第二层全连接：降维 4096→1024，还原到 Transformer 隐层维度
          )
          (layer_norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)        # Pre-Norm LN2：FFN 前的层归一化
        )
      )
    )
    (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)         # 最终层归一化：对 23 层 Encoder 输出做归一化，输出 [B, 77, 1024] 的文本特征序列
  )
)
```

## 三、分词器（Tokenizer）与 U-Net 结构

### 3.1 CLIPTokenizer 信息

`CLIPTokenizer` 使用 BPE（Byte Pair Encoding）算法对文本进行分词，关键参数：

| 参数 | 值 | 含义 |
|------|----|------|
| `vocab_size` | 49408 | 词表大小，包含 BPE 子词 token |
| `model_max_length` | 77 | 最大 token 序列长度，超出部分会被截断 |
| `padding_side` | `right` | 批内不足 77 个 token 时在右侧填充 |
| `bos_token` | `<\|startoftext\|>` | 序列开始标记（token id = 49406） |
| `eos_token` | `<\|endoftext\|>` | 序列结束标记（token id = 49407） |
| `pad_token` | `!` | 填充标记（token id = 0） |

```python
CLIPTokenizer(
  name_or_path='...tokenizer',              # 分词器权重的本地缓存路径（首次运行时从 HuggingFace Hub 下载）
  vocab_size=49408,                         # 词表大小：49408 个 BPE 子词 token（包含特殊 token）
  model_max_length=77,                      # 最大序列长度：77 个 token，超过则截断，不足则在右侧填充
  is_fast=False,                            # 使用 Python 实现的慢速分词器（非 Rust Fast Tokenizer）
  padding_side='right',                     # 填充方向：在序列右侧添加 pad_token 使批内长度一致
  truncation_side='right',                  # 截断方向：超过 77 个 token 时从右侧截断
  special_tokens={
    'bos_token': '<|startoftext|>',         # 序列开始标记（Beginning Of Sequence），每段文本前自动插入
    'eos_token': '<|endoftext|>',           # 序列结束标记（End Of Sequence），每段文本后自动插入
    'unk_token': '<|endoftext|>',           # 未知 token：词表外的字符映射到结束标记（id=49407）
    'pad_token': '!'                        # 填充标记：用于批内对齐，id=0（注意：不是空格，是感叹号）
  },
  added_tokens_decoder={
    0:     AddedToken("!"),                 # id=0 → 填充标记 "!"（pad_token）
    49406: AddedToken("<|startoftext|>"),   # id=49406 → 序列开始标记（bos_token）
    49407: AddedToken("<|endoftext|>"),     # id=49407 → 序列结束标记（eos_token / unk_token）
  }
)
```

### 3.2 U-Net（UNet2DConditionModel）结构

`UNet2DConditionModel` 是 SD 的核心去噪网络，以**文本嵌入为条件**对 latent 进行去噪。

**整体架构（以 SD 2.1-base 为例）：**
```
输入 latent [B, 4, 64, 64]
  ↓ conv_in → [B, 320, 64, 64]
  ↓ time_embedding: 时间步 → [B, 1280] 嵌入向量，注入各层
  ↓ down_blocks（编码器，逐步下采样）
      CrossAttnDownBlock2D × 3（含 Transformer2DModel 做交叉注意力）
      DownBlock2D × 1
  ↓ mid_block（瓶颈层）
      UNetMidBlock2DCrossAttn
  ↑ up_blocks（解码器，逐步上采样）
      UpBlock2D × 1
      CrossAttnUpBlock2D × 3
  ↑ conv_out → [B, 4, 64, 64]（预测的噪声）
```

**关键设计：**
- **交叉注意力（Cross-Attention）**：每个 Transformer 块的 `attn2` 中，
  Q 来自图像特征（如维度 320），K/V 来自文本嵌入（维度 1024），实现文本条件注入
- **跳跃连接（Skip Connection）**：编码器各层输出通过 ResnetBlock 的 `conv_shortcut` 连接到解码器
  （解码器 ResNet 输入通道数为编码器输出 + 上采样输出之和，如 2560 = 1280 + 1280）
- **时间步嵌入注入**：通过 `time_emb_proj` Linear 层将时间步嵌入加入每个 ResnetBlock

```python
UNet2DConditionModel(                                                                  # U-Net 条件去噪模型根类（以文本嵌入为条件）
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))        # 输入卷积：4 通道 latent 升至 320 通道；padding=1 保持空间 64×64 不变
  (time_proj): Timesteps()                                                              # 时间步投影：将整数时间步 t 转换为正弦余弦位置编码向量（维度 320）
  (time_embedding): TimestepEmbedding(                                                 # 时间步嵌入 MLP：通过两层全连接将 320 维位置编码映射为 1280 维嵌入，注入 U-Net 各层
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)                 # 第一层升维：320→1280
    (act): SiLU()                                                                      # SiLU 激活
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)                # 第二层：1280→1280，输出最终时间步嵌入，后续注入每个 ResNet 块
  )
  (down_blocks): ModuleList(                                                            # 编码器（下采样）：4 个块，通道 320→640→1280→1280，空间 64→32→16→8
    (0): CrossAttnDownBlock2D(                                                          # 第 1 个下采样块（含交叉注意力）：通道 320，空间 64→32
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(                                                 # 2 个 Transformer2D 模块，在每个 ResNet 之后处理 latent 特征
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)                           # Transformer 前的 GroupNorm
          (proj_in): Linear(in_features=320, out_features=320, bias=True)              # 空间特征展平后的投影（320→320）
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(                                                 # 基本 Transformer 块：自注意力 → 交叉注意力 → FFN
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)           # Pre-LN1，在自注意力前归一化
              (attn1): Attention(                                                       # 自注意力（Self-Attention）：Q/K/V 均来自 latent 特征，建立空间位置间的依赖
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_features=320, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=320, out_features=320, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((320,), eps=1e-05, elementwise_affine=True)           # Pre-LN2，在交叉注意力前归一化
              (attn2): Attention(                                                       # 交叉注意力（Cross-Attention）：Q 来自 latent，K/V 来自文本嵌入，实现文本条件注入
                (to_q): Linear(in_features=320, out_features=320, bias=False)          # Q：图像特征（320维）→ 查询向量
                (to_k): Linear(in_features=1024, out_features=320, bias=False)         # K：文本嵌入（1024维，来自 CLIPTextModel）→ 键向量（对齐到 320 维）
                (to_v): Linear(in_features=1024, out_features=320, bias=False)         # V：文本嵌入（1024维）→ 值向量（对齐到 320 维）
                (to_out): ModuleList(
                  (0): Linear(in_features=320, out_features=320, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((320,), eps=1e-05, elementwise_affine=True)           # Pre-LN3，在 FFN 前归一化
              (ff): FeedForward(                                                        # 前馈网络（FFN），使用 GEGLU 门控激活
                (net): ModuleList(
                  (0): GEGLU(                                                           # Gated GELU：将特征升维并分两路（一路 GELU 激活作门控），比普通 GELU 更强
                    (proj): Linear(in_features=320, out_features=2560, bias=True)      # 升维投影：320→2560（= 2×1280，分为 gate 和 value 两部分各 1280 维）
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=1280, out_features=320, bias=True)           # 降维投影：1280→320，GEGLU 门控后输出维度为 2560/2=1280
                )
              )
            )
          )
          (proj_out): Linear(in_features=320, out_features=320, bias=True)             # Transformer 输出投影（残差连接前）
        )
      )
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(                                                      # 2 个残差块：通道 320→320，每块内部通过 time_emb_proj 注入时间步信息
          (norm1): GroupNorm(32, 320, eps=1e-05, affine=True)
          (conv1): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=320, bias=True)       # 时间步注入：将 1280 维时间步嵌入投影为 320 维，加到特征图上
          (norm2): GroupNorm(32, 320, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
      (downsamplers): ModuleList(
        (0): Downsample2D(
          (conv): Conv2d(320, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))  # stride=2 下采样：64→32
        )
      )
    )
    (1): CrossAttnDownBlock2D(                                                          # 第 2 个下采样块：通道 320→640（升维），空间 32→16
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 640, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=640, out_features=640, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(                                                       # 自注意力，640 维特征
                (to_q): Linear(in_features=640, out_features=640, bias=False)
                (to_k): Linear(in_features=640, out_features=640, bias=False)
                (to_v): Linear(in_features=640, out_features=640, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=640, out_features=640, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (attn2): Attention(                                                       # 交叉注意力：K/V 来自 CLIP 文本嵌入（1024 维），Q 来自 latent（640 维）
                (to_q): Linear(in_features=640, out_features=640, bias=False)
                (to_k): Linear(in_features=1024, out_features=640, bias=False)         # 文本嵌入 1024 维→键向量 640 维
                (to_v): Linear(in_features=1024, out_features=640, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=640, out_features=640, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (ff): FeedForward(
                (net): ModuleList(
                  (0): GEGLU(
                    (proj): Linear(in_features=640, out_features=5120, bias=True)      # 升维：640→5120（= 2×2560）
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=2560, out_features=640, bias=True)           # 降维：2560→640
                )
              )
            )
          )
          (proj_out): Linear(in_features=640, out_features=640, bias=True)
        )
      )
      (resnets): ModuleList(
        (0): ResnetBlock2D(                                                             # 第 1 个残差块：通道升维 320→640，含 conv_shortcut 跳跃连接
          (norm1): GroupNorm(32, 320, eps=1e-05, affine=True)
          (conv1): Conv2d(320, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)) # 升维：320→640
          (time_emb_proj): Linear(in_features=1280, out_features=640, bias=True)
          (norm2): GroupNorm(32, 640, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(320, 640, kernel_size=(1, 1), stride=(1, 1))         # 跳跃连接升维：320→640
        )
        (1): ResnetBlock2D(                                                             # 第 2 个残差块：通道 640→640，无 conv_shortcut
          (norm1): GroupNorm(32, 640, eps=1e-05, affine=True)
          (conv1): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=640, bias=True)
          (norm2): GroupNorm(32, 640, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
      (downsamplers): ModuleList(
        (0): Downsample2D(
          (conv): Conv2d(640, 640, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))  # stride=2 下采样：32→16
        )
      )
    )
    (2): CrossAttnDownBlock2D(                                                          # 第 3 个下采样块：通道 640→1280，空间 16→8
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 1280, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=1280, out_features=1280, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=1280, out_features=1280, bias=False)
                (to_k): Linear(in_features=1280, out_features=1280, bias=False)
                (to_v): Linear(in_features=1280, out_features=1280, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=1280, out_features=1280, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (attn2): Attention(                                                       # 交叉注意力：K/V 来自文本（1024 维），Q 来自 latent（1280 维）
                (to_q): Linear(in_features=1280, out_features=1280, bias=False)
                (to_k): Linear(in_features=1024, out_features=1280, bias=False)
                (to_v): Linear(in_features=1024, out_features=1280, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=1280, out_features=1280, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (ff): FeedForward(
                (net): ModuleList(
                  (0): GEGLU(
                    (proj): Linear(in_features=1280, out_features=10240, bias=True)    # 升维：1280→10240（= 2×5120）
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=5120, out_features=1280, bias=True)          # 降维：5120→1280
                )
              )
            )
          )
          (proj_out): Linear(in_features=1280, out_features=1280, bias=True)
        )
      )
      (resnets): ModuleList(
        (0): ResnetBlock2D(                                                             # 通道升维 640→1280，含 conv_shortcut
          (norm1): GroupNorm(32, 640, eps=1e-05, affine=True)
          (conv1): Conv2d(640, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(640, 1280, kernel_size=(1, 1), stride=(1, 1))
        )
        (1): ResnetBlock2D(
          (norm1): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (conv1): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
      (downsamplers): ModuleList(
        (0): Downsample2D(
          (conv): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))  # stride=2 下采样：16→8（最小分辨率）
        )
      )
    )
    (3): DownBlock2D(                                                                   # 第 4 个下采样块（无交叉注意力）：通道 1280，空间 8 保持（最深处）
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(                                                      # 两个残差块，在最深处做特征整合（无 Transformer，纯 ResNet）
          (norm1): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (conv1): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
    )
  )
  (up_blocks): ModuleList(                                                              # 解码器（上采样）：4 个块，通道 2560→1280→640→320，空间 8→16→32→64
    (0): UpBlock2D(                                                                     # 第 1 个上采样块（无交叉注意力）：通道 2560→1280，空间 8→16
      (resnets): ModuleList(
        (0-2): 3 x ResnetBlock2D(                                                      # 3 个残差块；输入通道 2560 = 编码器输出1280 + skip连接1280（U-Net 跳跃连接拼接）
          (norm1): GroupNorm(32, 2560, eps=1e-05, affine=True)
          (conv1): Conv2d(2560, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  # 拼接后降维：2560→1280
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(2560, 1280, kernel_size=(1, 1), stride=(1, 1))       # 跳跃连接：2560→1280
        )
      )
      (upsamplers): ModuleList(
        (0): Upsample2D(                                                               # 双线性插值上采样：空间 8→16
          (conv): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
    )
    (1): CrossAttnUpBlock2D(                                                            # 第 2 个上采样块（含交叉注意力）：通道 2560→1280，空间 16→32
      (attentions): ModuleList(
        (0-2): 3 x Transformer2DModel(                                                 # 3 个 Transformer2D 模块
          (norm): GroupNorm(32, 1280, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=1280, out_features=1280, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=1280, out_features=1280, bias=False)
                (to_k): Linear(in_features=1280, out_features=1280, bias=False)
                (to_v): Linear(in_features=1280, out_features=1280, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=1280, out_features=1280, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (attn2): Attention(                                                       # 交叉注意力：解码器中继续注入文本条件
                (to_q): Linear(in_features=1280, out_features=1280, bias=False)
                (to_k): Linear(in_features=1024, out_features=1280, bias=False)
                (to_v): Linear(in_features=1024, out_features=1280, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=1280, out_features=1280, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
              (ff): FeedForward(
                (net): ModuleList(
                  (0): GEGLU(
                    (proj): Linear(in_features=1280, out_features=10240, bias=True)
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=5120, out_features=1280, bias=True)
                )
              )
            )
          )
          (proj_out): Linear(in_features=1280, out_features=1280, bias=True)
        )
      )
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(                                                      # 输入 2560=1280+1280（跳跃连接拼接），降维到 1280
          (norm1): GroupNorm(32, 2560, eps=1e-05, affine=True)
          (conv1): Conv2d(2560, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(2560, 1280, kernel_size=(1, 1), stride=(1, 1))
        )
        (2): ResnetBlock2D(                                                             # 输入 1920=1280+640（跳跃连接拼接），降维到 1280
          (norm1): GroupNorm(32, 1920, eps=1e-05, affine=True)
          (conv1): Conv2d(1920, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(1920, 1280, kernel_size=(1, 1), stride=(1, 1))
        )
      )
      (upsamplers): ModuleList(
        (0): Upsample2D(                                                               # 上采样：16→32
          (conv): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
    )
    (2): CrossAttnUpBlock2D(                                                            # 第 3 个上采样块：通道 1920→640，空间 32→64
      (attentions): ModuleList(
        (0-2): 3 x Transformer2DModel(
          (norm): GroupNorm(32, 640, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=640, out_features=640, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=640, out_features=640, bias=False)
                (to_k): Linear(in_features=640, out_features=640, bias=False)
                (to_v): Linear(in_features=640, out_features=640, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=640, out_features=640, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (attn2): Attention(
                (to_q): Linear(in_features=640, out_features=640, bias=False)
                (to_k): Linear(in_features=1024, out_features=640, bias=False)
                (to_v): Linear(in_features=1024, out_features=640, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=640, out_features=640, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
              (ff): FeedForward(
                (net): ModuleList(
                  (0): GEGLU(
                    (proj): Linear(in_features=640, out_features=5120, bias=True)
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=2560, out_features=640, bias=True)
                )
              )
            )
          )
          (proj_out): Linear(in_features=640, out_features=640, bias=True)
        )
      )
      (resnets): ModuleList(
        (0): ResnetBlock2D(                                                             # 输入 1920=1280+640（跳跃连接拼接），降维到 640
          (norm1): GroupNorm(32, 1920, eps=1e-05, affine=True)
          (conv1): Conv2d(1920, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=640, bias=True)
          (norm2): GroupNorm(32, 640, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(1920, 640, kernel_size=(1, 1), stride=(1, 1))
        )
        (1): ResnetBlock2D(                                                             # 输入 1280=640+640
          (norm1): GroupNorm(32, 1280, eps=1e-05, affine=True)
          (conv1): Conv2d(1280, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=640, bias=True)
          (norm2): GroupNorm(32, 640, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(1280, 640, kernel_size=(1, 1), stride=(1, 1))
        )
        (2): ResnetBlock2D(                                                             # 输入 960=640+320
          (norm1): GroupNorm(32, 960, eps=1e-05, affine=True)
          (conv1): Conv2d(960, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=640, bias=True)
          (norm2): GroupNorm(32, 640, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(960, 640, kernel_size=(1, 1), stride=(1, 1))
        )
      )
      (upsamplers): ModuleList(
        (0): Upsample2D(                                                               # 上采样：32→64（还原到原始 latent 分辨率）
          (conv): Conv2d(640, 640, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
    )
    (3): CrossAttnUpBlock2D(                                                            # 第 4 个上采样块（最浅层）：通道 960→320，空间 64（无 Upsample）
      (attentions): ModuleList(
        (0-2): 3 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Linear(in_features=320, out_features=320, bias=True)
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_features=320, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=320, out_features=320, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm2): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn2): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=1024, out_features=320, bias=False)
                (to_v): Linear(in_features=1024, out_features=320, bias=False)
                (to_out): ModuleList(
                  (0): Linear(in_features=320, out_features=320, bias=True)
                  (1): Dropout(p=0.0, inplace=False)
                )
              )
              (norm3): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (ff): FeedForward(
                (net): ModuleList(
                  (0): GEGLU(
                    (proj): Linear(in_features=320, out_features=2560, bias=True)
                  )
                  (1): Dropout(p=0.0, inplace=False)
                  (2): Linear(in_features=1280, out_features=320, bias=True)
                )
              )
            )
          )
          (proj_out): Linear(in_features=320, out_features=320, bias=True)
        )
      )
      (resnets): ModuleList(
        (0): ResnetBlock2D(                                                             # 输入 960=640+320（跳跃连接拼接），降维到 320
          (norm1): GroupNorm(32, 960, eps=1e-05, affine=True)
          (conv1): Conv2d(960, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=320, bias=True)
          (norm2): GroupNorm(32, 320, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(960, 320, kernel_size=(1, 1), stride=(1, 1))
        )
        (1-2): 2 x ResnetBlock2D(                                                      # 输入 640=320+320
          (norm1): GroupNorm(32, 640, eps=1e-05, affine=True)
          (conv1): Conv2d(640, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=1280, out_features=320, bias=True)
          (norm2): GroupNorm(32, 320, eps=1e-05, affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
          (conv_shortcut): Conv2d(640, 320, kernel_size=(1, 1), stride=(1, 1))
        )
      )
    )
  )

  (mid_block): UNetMidBlock2DCrossAttn(                                                # 瓶颈层（最小分辨率 8×8）：ResNet → 交叉注意力 Transformer → ResNet
    (attentions): ModuleList(
      (0): Transformer2DModel(
        (norm): GroupNorm(32, 1280, eps=1e-06, affine=True)
        (proj_in): Linear(in_features=1280, out_features=1280, bias=True)
        (transformer_blocks): ModuleList(
          (0): BasicTransformerBlock(
            (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
            (attn1): Attention(                                                         # 自注意力：在最小分辨率（8×8）建立 64 个像素点间的全局依赖
              (to_q): Linear(in_features=1280, out_features=1280, bias=False)
              (to_k): Linear(in_features=1280, out_features=1280, bias=False)
              (to_v): Linear(in_features=1280, out_features=1280, bias=False)
              (to_out): ModuleList(
                (0): Linear(in_features=1280, out_features=1280, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
            (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
            (attn2): Attention(                                                         # 交叉注意力：在瓶颈层注入文本条件（1024 维→1280 维）
              (to_q): Linear(in_features=1280, out_features=1280, bias=False)
              (to_k): Linear(in_features=1024, out_features=1280, bias=False)
              (to_v): Linear(in_features=1024, out_features=1280, bias=False)
              (to_out): ModuleList(
                (0): Linear(in_features=1280, out_features=1280, bias=True)
                (1): Dropout(p=0.0, inplace=False)
              )
            )
            (norm3): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
            (ff): FeedForward(
              (net): ModuleList(
                (0): GEGLU(
                  (proj): Linear(in_features=1280, out_features=10240, bias=True)
                )
                (1): Dropout(p=0.0, inplace=False)
                (2): Linear(in_features=5120, out_features=1280, bias=True)
              )
            )
          )
        )
        (proj_out): Linear(in_features=1280, out_features=1280, bias=True)
      )
    )
    (resnets): ModuleList(
      (0-1): 2 x ResnetBlock2D(                                                        # 夹住 Transformer 的两个残差块（瓶颈 ResNet→Attn→ResNet）
        (norm1): GroupNorm(32, 1280, eps=1e-05, affine=True)
        (conv1): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (time_emb_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (norm2): GroupNorm(32, 1280, eps=1e-05, affine=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (conv2): Conv2d(1280, 1280, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (nonlinearity): SiLU()
      )
    )
  )
  (conv_norm_out): GroupNorm(32, 320, eps=1e-05, affine=True)                          # 输出前的 GroupNorm 归一化
  (conv_act): SiLU()                                                                    # 输出前的 SiLU 激活
  (conv_out): Conv2d(320, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))       # 输出卷积：将 320 通道特征映射为 4 通道噪声预测，与输入 latent 维度完全一致
)
```

## 四、时间步嵌入函数实现与可视化

以下为 CompVis（SD 原始实现）中 `timestep_embedding` 函数的复现与可视化分析。
与 HuggingFace 版本的 `Timesteps()` 模块原理相同，均使用**正弦余弦位置编码**。

**为什么使用正弦余弦编码？**
- 不同时间步需要有区分度，且差异应随步骤差值单调变化
- 正弦编码天然周期性，低频分量捕捉整体趋势，高频分量区分相邻时间步
- 与 Transformer 位置编码同理：网络可以利用这些规律推断时间步之间的相对关系

In [ ]:
import torch                                                # 导入 PyTorch 库，提供张量操作和 exp/arange 等函数
import math                                                # 导入数学库，提供 log 函数用于计算频率衰减因子
import matplotlib.pyplot as plt                            # 导入 Matplotlib，用于绘制嵌入热力图和折线图


def timestep_embedding(timesteps, dim, max_period=10000, repeat_only=False):
    """
    生成正弦余弦时间步嵌入（Sinusoidal Timestep Embedding）。

    复现自 CompVis 原始代码，与 HuggingFace SD 中的 Timesteps() 模块原理相同。
    利用不同频率的正余弦函数对时间步进行编码，使 U-Net 感知当前去噪阶段。

    参数:
        timesteps (Tensor): 形状为 [N] 的一维张量，每个元素为一个时间步索引（可为浮点）。
                            - N: 批量中的时间步数量
        dim (int): 输出嵌入向量的维度 D，通常与 U-Net 隐藏层通道数一致。
        max_period (int): 控制最小频率的超参数，对应正弦编码公式中的最大周期（默认 10000）。
        repeat_only (bool): 若为 True，则通过重复时间步值填充维度（简化模式）。

    返回:
        Tensor: 形状为 [N, D] 的浮点嵌入张量。
                - N: 时间步数量（批量大小）
                - D: dim，前 D/2 维为余弦编码，后 D/2 维为正弦编码
    """
    if not repeat_only:                                     # 正常模式：使用正弦余弦编码生成嵌入（推荐用法）
        half = dim // 2                                     # 嵌入维度的一半，前半部分用余弦，后半部分用正弦
        freqs = torch.exp(                                  # 计算 half 个频率值：exp(-k·log(max_period)/half), k∈[0,half-1]
            -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half  # arange 生成 [0,1,...,half-1]，除以 half 归一化后乘以 -log(max_period)；形状 [half]
        ).to(device=timesteps.device)                       # 将频率张量迁移到与 timesteps 相同的设备（CPU/GPU）
        args = timesteps[:, None].float() * freqs[None]    # 计算相位：timesteps 扩展为 [N,1]，freqs 扩展为 [1,half]，广播相乘得 [N,half]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)  # 将余弦和正弦在最后一维拼接，形状 [N, dim]
        if dim % 2:                                         # 当 dim 为奇数时，余弦+正弦共 dim-1 维，需补一列 0 凑齐 dim 维
            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)  # 在末尾追加全零列，形状变为 [N, dim]
    else:                                                   # 简化模式（repeat_only=True）：将时间步值直接重复 dim 次填充（不常用）
        pass                                                # 此处可用 einops.repeat 实现：repeat(timesteps, 'b -> b d', d=dim)
    return embedding                                        # 返回时间步嵌入矩阵，形状 [N, D]


timesteps = torch.arange(0, 1000, 100)                     # 构造 10 个时间步索引：[0, 100, 200, ..., 900]，模拟等间距采样；形状 [10]
dim = 128                                                   # 嵌入维度，设为 128（前 64 维余弦，后 64 维正弦）

embeddings = timestep_embedding(timesteps, dim)             # 生成时间步嵌入矩阵，形状 [10, 128]：10 个时间步，每个 128 维

plt.figure(figsize=(12, 6))                                 # 创建 12×6 英寸的画布，容纳两个并排子图

plt.subplot(1, 2, 1)                                        # 创建第 1 个子图（1 行 2 列中的第 1 个）：热力图
plt.imshow(embeddings.cpu().numpy(), aspect='auto', cmap='viridis')  # 将 [10,128] 嵌入矩阵绘制为热力图，行=时间步索引，列=嵌入维度
plt.colorbar()                                              # 添加颜色条，显示数值到颜色的映射
plt.title('Timestep Embeddings Heatmap')                    # 设置子图标题
plt.xlabel('Embedding Dimension')                           # X 轴：嵌入维度（0~127）
plt.ylabel('Timestep Index')                                # Y 轴：时间步索引（0~9 对应 0~900）

plt.subplot(1, 2, 2)                                        # 创建第 2 个子图（1 行 2 列中的第 2 个）：折线图
selected_indices = [0, 3, 6, 9]                             # 选择 4 个时间步（索引 0,3,6,9，对应 t=0,300,600,900）展示其嵌入向量
for idx in selected_indices:                                # 遍历选中的时间步索引
    plt.plot(embeddings[idx].cpu().numpy(), label=f'Timestep {timesteps[idx].item()}')  # 绘制该时间步的 128 维嵌入向量折线图，X=维度索引，Y=嵌入值
plt.legend()                                                # 显示图例，区分不同时间步的曲线
plt.title('Selected Timestep Embeddings')                   # 设置子图标题
plt.xlabel('Embedding Dimension')                           # X 轴：嵌入维度（0~127）
plt.ylabel('Value')                                         # Y 轴：嵌入值（正余弦值，范围 [-1,1]）

plt.tight_layout()                                          # 自动调整子图间距，防止标题和标签重叠
plt.show()                                                  # 渲染并显示图形

## 五、Python 扩展解包（Extended Unpacking）语法演示

Python 的扩展解包语法 `a, b, *rest = iterable` 允许用 `*` 接收剩余元素，
在 SD 源码中常用于解包特征图的维度信息：`b, c, *spatial = x.shape`。

其中 `*spatial` 会将剩余所有维度（H、W 等空间尺寸）收集为一个列表，
便于后续处理不固定维度数量的张量（如 2D 图像或 3D 体素）。

In [ ]:
b, c, *spatial = 1, 2, 3, 4                                # 扩展解包：将元组 (1,2,3,4) 解包，b=1，c=2，*spatial 捕获剩余元素 [3,4]
                                                            # 对应 SD 源码中的 b, c, *spatial = x.shape（x 为特征图张量）
                                                            # b: 批量大小，c: 通道数，spatial: 空间维度列表（如 [H,W]=[3,4]）
print(b, c, spatial)                                        # 打印解包结果：b=1, c=2, spatial=[3,4]